# 03 · Feature Engineering — Technical Indicators and Lags
**Data Science Diploma · ENES UNAM León**

We build the **input variables (features)** for our models. The principle is to give the model past information so it can predict tomorrow's price.

### Features to create:
| Category | Feature | Description |
|---|---|---|
| **Lags** | Close_lag_1..7 | Closing price from the last N days |
| **Trend** | SMA_7, SMA_21 | Simple Moving Average (7 and 21 days) |
| **Trend** | EMA_12, EMA_26 | Exponential Moving Average |
| **Momentum** | RSI_14 | Relative Strength Index |
| **Momentum** | MACD, MACD_signal | Moving Average Convergence Divergence |
| **Volatility** | BB_upper, BB_lower | Bollinger Bands |
| **Volatility** | volatility_7d | 7-day return std dev |
| **Volume** | Volume_change | % change in volume |
| **Calendar** | day_of_week | Day of the week (0=Monday) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

df = pd.read_csv('../data/processed/sol_usd_eda.csv', index_col=0, parse_dates=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Data loaded: {df.shape}')
df.head(3)

## 3.1 Closing Price Lags

In [ ]:
# Lags: closing price from the last N days
# This tells the model: "today's price depends on previous prices"
for lag in [1, 2, 3, 5, 7, 14]:
    df[f'Close_lag_{lag}'] = df['Close'].shift(lag)

print('Lags created ✓')
df[['Close', 'Close_lag_1', 'Close_lag_2', 'Close_lag_7']].tail(5)

## 3.2 Moving Averages (SMA and EMA)

In [ ]:
# SMA — Simple Moving Average
df['SMA_7']  = df['Close'].rolling(window=7).mean()
df['SMA_21'] = df['Close'].rolling(window=21).mean()

# EMA — Exponential Moving Average (gives more weight to recent data)
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Close', color='gray', alpha=0.6, linewidth=1)
plt.plot(df.index, df['SMA_7'],  label='SMA 7',  color='#9945FF', linewidth=1.5)
plt.plot(df.index, df['SMA_21'], label='SMA 21', color='#14F195', linewidth=1.5)
plt.plot(df.index, df['EMA_12'], label='EMA 12', color='#FFB800', linewidth=1.5, linestyle='--')
plt.title('SOL-USD with Moving Averages')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/03_moving_averages.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 RSI — Relative Strength Index

Measures the speed and magnitude of price changes. Range 0–100:
- **> 70** → overbought (possible correction)
- **< 30** → oversold (possible bounce)

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=window - 1, min_periods=window).mean()
    avg_loss = loss.ewm(com=window - 1, min_periods=window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI_14'] = compute_rsi(df['Close'], 14)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['Close'], color='#9945FF', linewidth=1.5)
axes[0].set_title('Closing Price')
axes[0].set_ylabel('USD')

axes[1].plot(df.index, df['RSI_14'], color='#FFB800', linewidth=1.5)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].fill_between(df.index, 70, df['RSI_14'].clip(lower=70), alpha=0.2, color='red')
axes[1].fill_between(df.index, df['RSI_14'].clip(upper=30), 30, alpha=0.2, color='green')
axes[1].set_title('RSI (14 days)')
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/03_rsi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 MACD

In [ ]:
# MACD = EMA_12 - EMA_26
df['MACD']        = df['EMA_12'] - df['EMA_26']
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist']   = df['MACD'] - df['MACD_signal']

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['Close'], color='#9945FF', linewidth=1.5)
axes[0].set_title('Closing Price')
axes[0].set_ylabel('USD')

axes[1].plot(df.index, df['MACD'], label='MACD', color='#14F195', linewidth=1.5)
axes[1].plot(df.index, df['MACD_signal'], label='Signal', color='#FF4444', linewidth=1.5)
colors_macd = ['#14F195' if v >= 0 else '#FF4444' for v in df['MACD_hist'].fillna(0)]
axes[1].bar(df.index, df['MACD_hist'], color=colors_macd, alpha=0.5, label='Histogram')
axes[1].axhline(0, color='white', linewidth=0.5)
axes[1].set_title('MACD')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/03_macd.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Bollinger Bands

In [ ]:
window_bb = 20
df['BB_mid']   = df['Close'].rolling(window_bb).mean()
df['BB_std']   = df['Close'].rolling(window_bb).std()
df['BB_upper'] = df['BB_mid'] + 2 * df['BB_std']
df['BB_lower'] = df['BB_mid'] - 2 * df['BB_std']
df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['BB_mid']  # volatility feature

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Close', color='#9945FF', linewidth=1.5)
plt.plot(df.index, df['BB_upper'], label='Upper Band', color='red', linewidth=1, linestyle='--')
plt.plot(df.index, df['BB_mid'], label='SMA 20', color='white', linewidth=1, linestyle=':')
plt.plot(df.index, df['BB_lower'], label='Lower Band', color='green', linewidth=1, linestyle='--')
plt.fill_between(df.index, df['BB_lower'], df['BB_upper'], alpha=0.05, color='gray')
plt.title('Bollinger Bands (20 days, ±2σ)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/03_bollinger.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Volume and Calendar Features

In [ ]:
# Percentage change in volume
df['Volume_change'] = df['Volume'].pct_change() * 100
df['Volume_SMA7']   = df['Volume'].rolling(7).mean()

# Calendar features
df['day_of_week'] = df.index.dayofweek  # 0=Monday, 6=Sunday
df['month']       = df.index.month

print('Additional features created ✓')

## 3.7 Target Variable: Next Day's Price

In [ ]:
# Target: closing price of the NEXT day
# shift(-1) shifts the price one day forward (into the future)
df['Target'] = df['Close'].shift(-1)

print('Target variable (Target = next day Close) created ✓')
df[['Close', 'Target']].tail(5)

## 3.8 Final Cleanup — Remove NaN from Rolling Windows

In [ ]:
print(f'Rows before cleaning : {len(df)}')
print(f'NaN per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

df.dropna(inplace=True)

print(f'\nRows after cleaning : {len(df)}')
print('All columns have no NaN ✓')

## 3.9 Feature Summary and Correlation with Target

In [ ]:
feature_cols = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]

corr_target = df[feature_cols + ['Target']].corr()['Target'].drop('Target').sort_values(ascending=False)

plt.figure(figsize=(10, 7))
colors = ['#14F195' if v > 0 else '#FF4444' for v in corr_target]
corr_target.plot(kind='barh', color=colors, edgecolor='none')
plt.axvline(0, color='white', linewidth=0.8)
plt.title('Feature Correlation with Target (next day price)')
plt.xlabel("Pearson's Correlation")
plt.tight_layout()
plt.savefig('../data/raw/03_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(corr_target)

## 3.10 Save Final Dataset

In [ ]:
df.to_csv('../data/processed/sol_usd_features.csv')
print(f'Dataset with features saved ✓')
print(f'Final shape : {df.shape}')
print(f'Features    : {len(feature_cols)}')
print(f'Period      : {df.index.min().date()} → {df.index.max().date()}')

---
## ✅ Features Created

```python
FEATURE_COLS = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]
TARGET = 'Target'  # Next day's Close
```

**Next step →** `04_modeling.ipynb`